In [1]:
import pandas as pd
import json
from collections import defaultdict

In [2]:
csv_path = "../../data/twcs/twcs.csv"

In [19]:
df = df = pd.read_csv(csv_path, dtype={
    "tweet_id": str,
    "author_id": str,
    "response_tweet_id": str,
    "in_response_to_tweet_id": str
})


In [20]:
df

,tweet_id,author_id,inbound,created_at,text,response_tweet_id,in_response_to_tweet_id
0,1,sprintcare,False,Tue Oct 31 22:10:47 +0000 2017,@115712 I understand. I would like to assist y...,2,3
1,2,115712,True,Tue Oct 31 22:11:45 +0000 2017,@sprintcare and how do you propose we do that,NaN,1
2,3,115712,True,Tue Oct 31 22:08:27 +0000 2017,@sprintcare I have sent several private messag...,1,4
3,4,sprintcare,False,Tue Oct 31 21:54:49 +0000 2017,@115712 Please send us a Private Message so th...,3,5
4,5,115712,True,Tue Oct 31 21:49:35 +0000 2017,@sprintcare I did.,4,6
...,...,...,...,...,...,...,...
2811769,2987947,sprintcare,False,Wed Nov 22 08:43:51 +0000 2017,"@823869 Hey, we'd be happy to look into this f...",NaN,2987948
2811770,2987948,823869,True,Wed Nov 22 08:35:16 +0000 2017,@115714 wtf!? I’ve been having really shitty s...,2987947,NaN
2811771,2812240,121673,True,Thu Nov 23 04:13:07 +0000 2017,@143549 @sprintcare You have to go to https://...,NaN,2812239
2811772,2987949,AldiUK,False,Wed Nov 22 08:31:24 +0000 2017,"@823870 Sounds delicious, Sarah! 😋 https://t.c...",NaN,2987950


In [21]:
# Convert NaN to empty string
df["in_response_to_tweet_id"] = df["in_response_to_tweet_id"].fillna("")
df["response_tweet_id"] = df["response_tweet_id"].fillna("")

# Index by tweet_id for fast lookup
tweets = {str(row.tweet_id): row for _, row in df.iterrows()}

In [22]:
def find_root(tweet_id):
    """Traverse upward until no parent exists."""
    current = tweet_id
    while True:
        parent = str(tweets[current].in_response_to_tweet_id)
        if parent == "" or parent not in tweets:
            return current
        current = parent

In [24]:
# Map each root to list of tweet_ids
threads = defaultdict(list)

for tweet_id in tweets.keys():
    root = find_root(tweet_id)
    threads[root].append(tweet_id)

In [26]:
len(threads)

798197

In [27]:
def format_role(row):
    return "customer" if row.inbound else "support"

In [37]:
output = []

for root_id, tweet_ids in threads.items():
    # Sort tweets in this thread by time
    items = [tweets[i] for i in tweet_ids]
    items = sorted(items, key=lambda x: x.created_at)

    # Build conversation list
    conversation = []
    last_brand = None
    
    for row in items:
        role = format_role(row)
        # conversation.append({
        #     "role": role,
        #     "text": row.text
        # })
        conversation.append({
            "role": role,
            "text": row.text,
            "author_id": row.author_id
        })


        # brand = author if support tweet
        if role == "support":
            last_brand = row.author_id

    # Build RAG text
    # rag_text = "\n".join([
    #     f"[{msg['role'].capitalize()}] {msg['text']}"
    #     for msg in conversation
    # ])

    rag_text = "\n".join([
        f"[{msg['author_id']}] {msg['text']}"
        for msg in conversation
    ])


    output.append({
        "thread_id": root_id,
        "brand": last_brand,  # could be None if no support msg
        "start_time": str(items[0].created_at),
        "messages": conversation,
        "text_for_rag": rag_text,
        "tweet_ids": tweet_ids
    })

In [38]:
output[0]

{'thread_id': '8',
 'brand': 'sprintcare',
 'start_time': 'Tue Oct 31 21:45:10 +0000 2017',
 'messages': [{'role': 'customer',
   'text': '@sprintcare is the worst customer service',
   'author_id': '115712'},
  {'role': 'support',
   'text': '@115712 Hello! We never like our customers to feel like they are not valued.',
   'author_id': 'sprintcare'},
  {'role': 'support',
   'text': '@115712 I would love the chance to review the account and provide assistance.',
   'author_id': 'sprintcare'},
  {'role': 'support',
   'text': '@115712 Can you please send us a private message, so that I can gain further details about your account?',
   'author_id': 'sprintcare'},
  {'role': 'customer',
   'text': '@sprintcare the only way I can get a response is to tweet apparently',
   'author_id': '115712'},
  {'role': 'customer', 'text': '@sprintcare I did.', 'author_id': '115712'},
  {'role': 'support',
   'text': '@115712 Please send us a Private Message so that we can further assist you. Just clic

In [ ]:
rag_texts_output = [
    {
        "_id": item["thread_id"],
        "title": f"{item['brand']}_{item['start_time']}",
        "text": item["text_for_rag"],
        "metadata":{
            "tweet_ids": item["tweet_ids"]
        }
    }
    for item in output]

In [42]:
with open("threads.jsonl", "w", encoding="utf-8") as f:
    for doc in rag_texts_output:
        f.write(json.dumps(doc, ensure_ascii=False) + "\n")

print("Done! Generated threads.jsonl with", len(rag_texts_output), "threads.")

Done! Generated threads.jsonl with 798197 threads.
